# Numverify — Phone Number Validation

Demonstrates **numverify_validate** and **numverify_countries** tools
using the [Numverify API](https://numverify.com).

Requires a `NUMVERIFY_ACCESS_KEY` environment variable set in `.env`.

---
## Setup

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from demos.shared.llm_setup import get_llm
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from pymcpx.numverify import NumverifyValidateTool, NumverifyCountriesTool

llm = get_llm()

tools = [NumverifyValidateTool(), NumverifyCountriesTool()]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a phone number validation assistant. Use the Numverify tools to "
        "validate phone numbers and look up supported countries.\n\n"
        "Available tools:\n"
        "- numverify_validate(number, country_code) \u2014 validate a phone number and get carrier, location, line type\n"
        "- numverify_countries() \u2014 list all supported countries and dialing codes",
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

### Scenario 1: Validate a US Phone Number

Check the carrier, location, and line type for a US mobile number.

In [ ]:
result = executor.invoke({"input": "Validate the phone number +14158586273. What carrier does it belong to and what type of line is it?"})
print(result["output"])

### Scenario 2: Validate an International Number with Country Code

Validate a UK number using national format with the country code hint.

In [ ]:
result = executor.invoke({"input": "Check if 2079460956 is a valid UK phone number. The country code is GB."})
print(result["output"])

### Scenario 3: List Supported Countries

Get the full list of countries and dialing codes supported by Numverify.

In [ ]:
result = executor.invoke({"input": "Show me all supported countries and their dialing codes."})
print(result["output"])